# 02 — EDA: Prediccion de Diagnostico de Cancer

**Proyecto:** UAX Inteligencia Artificial 2025-2026  
**Fecha:** 2026-05-04  

Este notebook documenta el EDA completo sobre las 6 fuentes de datos unidas por `paciente_id`.  
Entregables generados en la misma ejecucion:
- `data/interim/joined.csv` — dataset unido sin imputacion (50 001 filas x 38 columnas)
- `reports/eda_report.json` — resumen estadistico maquina-legible
- `reports/figures/fig01-fig10` — figuras del analisis

**No se realiza imputation, split ni escalado** — esas operaciones corresponden a `src/features/preprocess.py` post-split.


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless; cambiar a 'inline' en Jupyter interactivo
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

BASE  = Path('/Users/barrechee/School/Universidad/3/UAX/2/Inteligencia-Artificial/Casos/Prediccion-Cancer')
RAW   = BASE / 'data' / 'raw'
INTER = BASE / 'data' / 'interim'
FIG   = BASE / 'reports' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
print('Environment ready.')


Environment ready.


## 1. Carga de los 6 CSV


In [2]:
names = ['bioquimicos', 'clinicos', 'geneticos', 'economicos', 'generales', 'sociodemografico']
dfs   = {n: pd.read_csv(RAW / f'{n}.csv') for n in names}

print('=' * 65)
print(f"  {'Fuente':<22} {'Filas':>7} {'Cols':>5} {'IDs unicos':>12} {'Nulos':>7}")
print('=' * 65)
for n, df in dfs.items():
    print(f"  {n:<22} {len(df):>7,} {len(df.columns):>5} "
          f"{df['paciente_id'].nunique():>12,} {df.isnull().sum().sum():>7}")
print('=' * 65)


  Fuente                   Filas  Cols   IDs unicos   Nulos
  bioquimicos             50,001     8       50,001       0
  clinicos                50,001     8       50,001       0
  geneticos               50,001     8       50,001       0
  economicos              50,001     6       50,001       0
  generales               50,001     5       50,001       0
  sociodemografico        50,001     8       50,001       0


## 2. Union por `paciente_id` (left join sobre `clinicos`)

La tabla principal es `clinicos` porque contiene la variable objetivo `cancer`.  
Las 5 tablas restantes se unen con `how='left'`. Dado que los 6 CSV comparten exactamente los mismos 50 001 `paciente_id`, el resultado es identico a un inner join o un outer join (cobertura 100%).


In [3]:
joined = dfs['clinicos'].copy()
for n in names:
    if n != 'clinicos':
        joined = joined.merge(dfs[n], on='paciente_id', how='left')

print(f'Shape joined: {joined.shape}')
print(f'paciente_id unicos: {joined["paciente_id"].nunique():,}')
print(f'Nulos tras join: {joined.isnull().sum().sum()}')
print(f'Columnas ({len(joined.columns)}):')
print(list(joined.columns))


Shape joined: (50001, 38)
paciente_id unicos: 50,001
Nulos tras join: 0
Columnas (38):
['paciente_id', 'diabetes', 'hipertension', 'obesidad', 'cancer', 'enfermedad_cardiaca', 'asma', 'epoc', 'glucosa', 'colesterol', 'trigliceridos', 'hemoglobina', 'leucocitos', 'plaquetas', 'creatinina', 'mut_BRCA1', 'mut_TP53', 'mut_EGFR', 'mut_KRAS', 'mut_PIK3CA', 'mut_ALK', 'mut_BRAF', 'tipo_seguro', 'coste_total', 'coste_farmaco', 'num_ingresos', 'dias_hospital', 'fumador', 'alcohol', 'actividad_fisica', 'vive', 'edad', 'nivel_educativo', 'nivel_ingresos', 'zona', 'estado_civil', 'num_hijos', 'distancia_hospital_km']


## 3. Analisis de la variable objetivo `cancer`


In [4]:
n_pos = int(joined['cancer'].sum())
n_neg = len(joined) - n_pos
prev  = joined['cancer'].mean()
ratio = n_neg / n_pos

print(f'N total     : {len(joined):,}')
print(f'cancer = 1  : {n_pos:,} ({prev*100:.2f}%)')
print(f'cancer = 0  : {n_neg:,} ({(1-prev)*100:.2f}%)')
print(f'Ratio 1:N   : 1 : {ratio:.2f}')

if prev < 0.01:
    print('WARNING: prevalencia < 1% — desbalance extremo')
elif prev > 0.5:
    print('WARNING: prevalencia > 50%')
else:
    print(f'INFO: desbalance moderado (1:{ratio:.1f}), manejable con class_weight')


N total     : 50,001
cancer = 1  : 9,644 (19.29%)
cancer = 0  : 40,357 (80.71%)
Ratio 1:N   : 1 : 4.18
INFO: desbalance moderado (1:4.2), manejable con class_weight


## 4. Estadistica descriptiva de features numericas


In [5]:
numeric_cols = joined.select_dtypes('number').columns.tolist()
feature_num  = [c for c in numeric_cols if c != 'cancer']

desc = joined[feature_num].describe().T
desc['null_pct'] = (joined[feature_num].isnull().sum() / len(joined) * 100).round(2)
print(desc[['count','mean','std','min','25%','50%','75%','max','null_pct']].to_string())


                         count          mean           std     min      25%      50%       75%        max  null_pct
diabetes               50001.0      0.344653      0.475260    0.00     0.00     0.00      1.00       1.00       0.0
hipertension           50001.0      0.443231      0.496772    0.00     0.00     0.00      1.00       1.00       0.0
obesidad               50001.0      0.354193      0.478273    0.00     0.00     0.00      1.00       1.00       0.0
enfermedad_cardiaca    50001.0      0.166277      0.372333    0.00     0.00     0.00      0.00       1.00       0.0
asma                   50001.0      0.081598      0.273755    0.00     0.00     0.00      0.00       1.00       0.0
epoc                   50001.0      0.091818      0.288772    0.00     0.00     0.00      0.00       1.00       0.0
glucosa                50001.0    102.191963     19.121454   55.00    88.73   101.78    115.25     179.23       0.0
colesterol             50001.0    193.660843     32.459019  120.00   171

## 5. Columnas constantes y casi-constantes


In [6]:
cat_cols = [c for c in joined.columns
            if joined[c].dtype == object or str(joined[c].dtype) == 'string'
            and c != 'paciente_id']

print('Columnas con >99% mismo valor:')
found = False
for c in feature_num + cat_cols:
    vc = joined[c].value_counts(normalize=True)
    if vc.iloc[0] >= 0.99:
        print(f'  {c}: top={vc.index[0]!r}, {vc.iloc[0]*100:.1f}%')
        found = True
if not found:
    print('  (ninguna excepto alcohol, verificado abajo)')

print(f'\nalcohol value_counts: {joined["alcohol"].value_counts().to_dict()}')
print('=> alcohol es CONSTANTE (100% = 1). Se debe EXCLUIR antes de modelar.')


Columnas con >99% mismo valor:
  alcohol: top=np.int64(1), 100.0%

alcohol value_counts: {1: 50001}
=> alcohol es CONSTANTE (100% = 1). Se debe EXCLUIR antes de modelar.


## 6. Correlacion Pearson con el target (numericas)


In [7]:
corr_s = joined[feature_num].corrwith(joined['cancer']).sort_values(key=abs, ascending=False)

leakage_set = {'coste_total','coste_farmaco','num_ingresos','dias_hospital','vive'}
print('Correlacion Pearson con cancer:')
for feat, val in corr_s.items():
    flag = '  <-- LEAKAGE POTENCIAL' if feat in leakage_set else ''
    print(f'  {feat:<30} {val:+.4f}{flag}')


Correlacion Pearson con cancer:
  coste_total                    +0.8907  <-- LEAKAGE POTENCIAL
  dias_hospital                  +0.8777  <-- LEAKAGE POTENCIAL
  coste_farmaco                  +0.8531  <-- LEAKAGE POTENCIAL
  num_ingresos                   +0.6435  <-- LEAKAGE POTENCIAL
  vive                           -0.3542  <-- LEAKAGE POTENCIAL
  mut_BRCA1                      +0.2186
  fumador                        +0.2167
  obesidad                       +0.1977
  mut_TP53                       +0.1871
  mut_KRAS                       +0.1665
  glucosa                        +0.1511
  trigliceridos                  +0.1086
  hipertension                   +0.1007
  mut_EGFR                       +0.0996
  leucocitos                     +0.0983
  colesterol                     +0.0874
  diabetes                       +0.0760
  hemoglobina                    -0.0734
  mut_PIK3CA                     +0.0676
  edad                           +0.0542
  mut_BRAF                       

/opt/anaconda3/envs/uax-tf/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/anaconda3/envs/uax-tf/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## 7. Lift de variables categoricas frente a `cancer`


In [8]:
for feat in ['actividad_fisica','tipo_seguro','nivel_educativo',
             'nivel_ingresos','zona','estado_civil','fumador']:
    lift = joined.groupby(feat)['cancer'].mean().sort_values(ascending=False) * 100
    print(f'\n{feat}:')
    for cat, val in lift.items():
        print(f'  {str(cat):<25} P(cancer=1) = {val:.2f}%')



actividad_fisica:
  Baja                      P(cancer=1) = 24.02%
  Moderada                  P(cancer=1) = 17.19%
  Alta                      P(cancer=1) = 12.30%

tipo_seguro:
  Privado                   P(cancer=1) = 31.94%
  Mixto                     P(cancer=1) = 19.53%
  Publico                   P(cancer=1) = 13.83%

nivel_educativo:
  Sin estudios              P(cancer=1) = 19.73%
  Universitario             P(cancer=1) = 19.34%
  Secundaria                P(cancer=1) = 19.34%
  Primaria                  P(cancer=1) = 18.98%

nivel_ingresos:
  Medio                     P(cancer=1) = 19.40%
  Muy bajo                  P(cancer=1) = 19.33%
  Alto                      P(cancer=1) = 19.32%
  Bajo                      P(cancer=1) = 19.09%

zona:
  Rural                     P(cancer=1) = 19.45%
  Semiurbana                P(cancer=1) = 19.35%
  Urbana                    P(cancer=1) = 19.20%

estado_civil:
  Viudo                     P(cancer=1) = 20.04%
  Soltero                   

## 8. Generacion de figuras

Se generan 10 figuras PNG en `reports/figures/`. Para verlas en Jupyter interactivo, cambiar `matplotlib.use('Agg')` por `%matplotlib inline`.


In [9]:
# Ejecutar reports/generate_eda_figures.py para regenerar las figuras
import subprocess, sys
result = subprocess.run(
    [sys.executable, str(BASE / 'reports' / 'generate_eda_figures.py')],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)


Dataset: (50001, 38), cancer=1: 9644 (19.29%), ratio 1:4.18
fig01 saved
fig02 saved
fig03 saved
fig04 saved
fig05 saved
fig06 saved
fig07 saved
fig08 saved
fig09 saved
fig10 saved

JSON report saved: /Users/barrechee/School/Universidad/3/UAX/2/Inteligencia-Artificial/Casos/Prediccion-Cancer/reports/eda_report.json

All outputs complete.
Figures: ['fig01_target_distribution.png', 'fig02_bioquimicos_dist.png', 'fig03_geneticos_lift.png', 'fig04_clinicos_lift.png', 'fig05_categorical_lift.png', 'fig06_corr_heatmap.png', 'fig07_leakage_warning.png', 'fig08_sociodem_num.png', 'fig09_economicos_leakage.png', 'fig10_feature_signal.png', 'test.png']



## 9. Persistencia del dataset unido


In [10]:
out = INTER / 'joined.csv'
joined.to_csv(str(out), index=False)
print(f'Guardado: {out}')
print(f'Shape: {joined.shape}')
print(f'Nulos: {joined.isnull().sum().sum()}')
print(f'Dtypes:\n{joined.dtypes.value_counts()}')
joined.head(3)


Guardado: /Users/barrechee/School/Universidad/3/UAX/2/Inteligencia-Artificial/Casos/Prediccion-Cancer/data/interim/joined.csv
Shape: (50001, 38)
Nulos: 0
Dtypes:
int64      21
float64    10
str         7
Name: count, dtype: int64


,paciente_id,diabetes,hipertension,obesidad,cancer,enfermedad_cardiaca,asma,epoc,glucosa,colesterol,...,alcohol,actividad_fisica,vive,edad,nivel_educativo,nivel_ingresos,zona,estado_civil,num_hijos,distancia_hospital_km
0,P1000000,0,0,1,0,0,0,0,94.66,205.32,...,1,Moderada,1,53,Secundaria,Medio,Urbana,Casado,2,25.6
1,P1000001,0,0,1,0,0,0,0,103.94,235.17,...,1,Moderada,1,66,Secundaria,Alto,Urbana,Divorciado,0,15.2
2,P1000002,1,1,0,1,0,1,0,131.34,138.42,...,1,Moderada,1,33,Secundaria,Bajo,Urbana,Casado,3,43.7


## 10. Resumen EDA — hallazgos clave para modelado

### Variable objetivo
- N = 50 001 pacientes, `cancer=1` en **9 644 casos (19.29%)**
- Ratio desbalance 1:4.18 — **moderado, manejable con `class_weight`**

### Leakage confirmado — EXCLUIR sin excepcion
| Variable | r con cancer | Motivo |
|---|:---:|---|
| `coste_total` | 0.891 | Consecuencia del diagnostico, no causa |
| `dias_hospital` | 0.878 | Idem |
| `coste_farmaco` | 0.853 | Idem |
| `num_ingresos` | 0.644 | Idem |
| `vive` | -0.354 | Resultado vital post-diagnostico |
| `alcohol` | ~0.000 | Constante, sin varianza |

### Top features por senial lineal (sin leakage)
| Feature | r | Fuente | Tratamiento propuesto |
|---|:---:|---|---|
| `mut_BRCA1` | 0.219 | geneticos | Binaria, usar tal cual |
| `fumador` | 0.217 | generales | Binaria, usar tal cual |
| `obesidad` | 0.198 | clinicos | Binaria, valorar leakage indirecto |
| `mut_TP53` | 0.187 | geneticos | Binaria, usar tal cual |
| `mut_KRAS` | 0.167 | geneticos | Binaria, usar tal cual |
| `glucosa` | 0.151 | bioquimicos | Continua, StandardScaler post-split |
| `actividad_fisica` | ~0.12 | generales | Categorica, OHE o ordinal |
| `trigliceridos` | 0.109 | bioquimicos | Continua, StandardScaler post-split |
| `hipertension` | 0.101 | clinicos | Binaria, valorar leakage indirecto |
| `mut_EGFR` | 0.100 | geneticos | Binaria, usar tal cual |

### Proximos pasos
1. Split estratificado 80/20 con `stratify=cancer`, seed=42 (`src/features/split.py`)
2. Imputacion (no hay nulos, pero se codificara) + OHE + StandardScaler sobre train, aplicar a test (`src/features/preprocess.py`)
3. Modelos clasicos: LR, RF, XGBoost, LightGBM con metricas F1 y AUC-ROC (`ml-classical`)
4. MLP con `class_weight`, ajuste de umbral sobre validacion (`mlp-designer`)
